# Notebook 04 — Frozen test confirmation

Select **Runtime → Change runtime type → T4 GPU**. Run the single code cell below. Keep the existing `SensorRepair_ICDSIS` Drive project and all Notebook 01–03 outputs. No dataset download, retraining, or manual file movement is needed.

This freezes the current fold-0 models, selected thresholds, primary comparator, and evaluation plan before computing test predictions. It evaluates clean data, nine known-fault conditions, and three held-out stuck-fault conditions separately. All original baselines and frozen tuned baselines are retained. The label-assisted oracle is diagnostic only.

Expect HAR to match keep: its selected utility threshold of 2 disables all interventions. The notebook does not seek new settings on test data. If interrupted, rerun the same cell in the same environment to reuse completed condition caches.

**Scope:** one seed, one fold. DSADS has only two test participants here, HAR nine. Results are descriptive and do not establish statistical significance, population generalization, or novelty. The full protocol still requires remaining DSADS folds and repeated seeds. After inspecting these results, they cannot serve as fresh confirmation of a revised method.

The code checks prior-file hashes, neural-output parity, saved policy probabilities, and validation decisions before test inference. It stops on incompatibility rather than refitting. A software-version mismatch message states the version used by Notebook 03.

At completion, share the printed output and the three named summary files. Timings include candidate generation separately from policy-head scoring; intervention rate is not a compute-saving claim.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import gc
import hashlib
import json
import shutil
import time
import zlib
import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from threadpoolctl import threadpool_limits

PROJECT = Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT = PROJECT / 'outputs' / 'notebook04_v1'
LOCAL = Path('/content/sensorrepair_notebook04_v1')
SEED = 2026
FOLD = 'fold_0'
INFERENCE_BATCH_SIZE = 128
REQUIRE_GPU = True
CODE_VERSION = 'sensorrepair_nb04_v1'
CELL_SIGNATURE = '2b08d4143ff2ce51342b88fdd79687bafa28dcfc421a0d115e0e1fbbb559db99'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and DEVICE.type != 'cuda':
    raise RuntimeError('Choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')
OUTPUT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = False
print('NOTEBOOK 04 | frozen fold_0 test confirmation | one executable cell', flush=True)
print('Device:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU', '| PyTorch:', torch.__version__)
print('No training or threshold selection. Test participants; known faults and held-out stuck faults reported separately.', flush=True)




def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def cuda_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


class ActivityCNN(nn.Module):
    def __init__(self, channels, classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 7, padding=3), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.GroupNorm(8, 96), nn.GELU(),
            nn.Conv1d(96, 128, 5, stride=2, padding=2), nn.GroupNorm(8, 128), nn.GELU(),
            nn.AvgPool1d(kernel_size=8, stride=8), nn.Flatten(),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, classes))

    def forward(self, values, available=None):
        if available is None:
            available = torch.ones_like(values)
        return self.encoder(torch.cat([values, available], dim=1))


class SensorReconstructor(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 5, padding=2), nn.GELU(),
            nn.Conv1d(64, 96, 5, padding=2), nn.GELU(),
            nn.Conv1d(96, 64, 5, padding=2), nn.GELU(), nn.Conv1d(64, channels, 1))

    def forward(self, values, available):
        return self.network(torch.cat([values, available], dim=1))


def corrupt_signals(values, groups, rng, fault=None, severity=None):
    # Fault locations and severity metadata are never returned as model features.
    result = np.array(values, dtype=np.float32, copy=True)
    for i in range(len(result)):
        kind = fault if fault is not None else str(rng.choice(['dropout', 'bias', 'gain']))
        level = int(severity) if severity is not None else int(rng.integers(0, 3))
        group = groups[int(rng.integers(len(groups)))]['channels']
        if kind == 'dropout':
            fraction = protocol['fault_severities']['dropout']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            result[i, group, start:start + length] = 0.0
        elif kind == 'bias':
            amount = protocol['fault_severities']['bias']['absolute_offset_in_training_std'][level]
            signs = rng.choice([-1.0, 1.0], size=len(group)).astype(np.float32)
            result[i, group, :] += np.float32(amount) * signs[:, None]
        elif kind == 'gain':
            factor = float(protocol['fault_severities']['gain']['factor'][level])
            factor = factor if rng.random() < 0.5 else 1.0 / factor
            result[i, group, :] *= np.float32(factor)
        elif kind == 'stuck':
            fraction = protocol['fault_severities']['stuck']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            first = result[i, group, start].copy()
            result[i, group, start:start + length] = first[:, None]
        else:
            raise ValueError(f'Unknown fault: {kind}')
    return result


@torch.inference_mode()
def action_outputs(classifier, reconstructor, values, groups, classes):
    classifier.eval()
    reconstructor.eval()
    number = len(values)
    group_count = len(groups)
    probabilities = np.empty((number, 1 + 2 * group_count, classes), dtype=np.float32)
    residuals = np.empty((number, group_count), dtype=np.float32)
    group_statistics = np.empty((number, group_count, 4), dtype=np.float32)
    cuda_sync()
    started = time.perf_counter()
    for start in range(0, number, INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, number)
        observed = torch.as_tensor(values[start:stop], device=DEVICE)
        probabilities[start:stop, 0] = classifier(observed).softmax(dim=1).cpu().numpy()
        for index, group in enumerate(groups):
            channels = group['channels']
            masked, available = observed.clone(), torch.ones_like(observed)
            masked[:, channels, :], available[:, channels, :] = 0.0, 0.0
            reconstruction = reconstructor(masked, available)
            repaired = observed.clone()
            repaired[:, channels, :] = reconstruction[:, channels, :]
            probabilities[start:stop, 1 + index] = classifier(masked, available).softmax(dim=1).cpu().numpy()
            probabilities[start:stop, 1 + group_count + index] = classifier(repaired).softmax(dim=1).cpu().numpy()
            residuals[start:stop, index] = (observed[:, channels, :] - reconstruction[:, channels, :]).square().mean(dim=(1, 2)).cpu().numpy()
            signal = values[start:stop, channels, :]
            group_statistics[start:stop, index] = np.stack([
                np.abs(signal).mean(axis=(1, 2)), signal.std(axis=(1, 2)),
                np.abs(np.diff(signal, axis=2)).mean(axis=(1, 2)),
                (signal == 0).mean(axis=(1, 2))], axis=1)
    cuda_sync()
    seconds = time.perf_counter() - started
    assert np.isfinite(probabilities).all() and np.isfinite(residuals).all()
    assert np.allclose(probabilities.sum(axis=2), 1.0, atol=1e-5)
    return probabilities, residuals, group_statistics, seconds


def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)


def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names


def predict_head(model, X):
    if isinstance(model, dict):
        return np.full(len(X), model['constant_probability'], dtype=np.float32)
    indices = np.flatnonzero(model.classes_ == 1)
    return model.predict_proba(X)[:, int(indices[0])].astype(np.float32) if len(indices) else np.zeros(len(X), dtype=np.float32)


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def evaluate_choices(pack, choices, method):
    p, y = pack['probabilities'], pack['y']
    D, N, A, K = p.shape
    assert choices.shape == (D, N) and choices.min() >= 0 and choices.max() < A
    predictions = p.argmax(axis=-1)
    original_correct = predictions[:, :, 0] == y[None, :]
    selected = np.take_along_axis(predictions, choices[:, :, None], axis=2).squeeze(2)
    rows = []
    for i, condition in enumerate(pack['conditions']):
        correct = selected[i] == y
        helpful = float((~original_correct[i] & correct).mean())
        harmful = float((original_correct[i] & ~correct).mean())
        accuracy = float(correct.mean())
        assert abs(accuracy - original_correct[i].mean() - helpful + harmful) < 1e-10
        rows.append({'condition': condition, 'method': method, 'windows': N,
                     'macro_f1': macro_f1(y, selected[i], K), 'accuracy': accuracy,
                     'intervention_rate': float((choices[i] != 0).mean()), 'helpful_rate': helpful,
                     'harmful_rate': harmful,
                     'harm_given_original_correct': float(np.sum(original_correct[i] & ~correct) / max(1, original_correct[i].sum()))})
    return rows


def baseline_choices(pack, scale):
    p, residual = pack['probabilities'], pack['residuals']
    D, N, A, K = p.shape
    G = residual.shape[-1]
    ratio = residual / scale[None, None, :]
    suspect = ratio.argmax(axis=-1)
    gate = ratio.max(axis=-1) > 1.0
    original_correct = p.argmax(axis=-1) == pack['y'][None, :, None]
    return {
        'keep': np.zeros((D, N), dtype=np.int64),
        'mask_suspect': 1 + suspect,
        'reconstruct_suspect': 1 + G + suspect,
        'residual_gate_mask': np.where(gate, 1 + suspect, 0),
        'residual_gate_reconstruct': np.where(gate, 1 + G + suspect, 0),
        'confidence_select': p.max(axis=-1).argmax(axis=-1),
        'oracle_diagnostic': original_correct.argmax(axis=-1)
    }


def choose_actions(pack, probability_fix, probability_harm, scale, settings):
    family = settings['family']
    D, N, A, K = pack['probabilities'].shape
    if family in ['harm_aware_policy', 'benefit_only_policy']:
        scores = probability_fix - settings['harm_penalty'] * probability_harm
        selected = scores.argmax(axis=-1)
        return np.where(scores.max(axis=-1) > settings['threshold'], selected + 1, 0)
    if family == 'confidence_margin_tuned':
        confidence = pack['probabilities'].max(axis=-1)
        selected = confidence.argmax(axis=-1)
        return np.where(confidence.max(axis=-1) > confidence[:, :, 0] + settings['threshold'], selected, 0)
    if settings['threshold'] is None:
        return np.zeros((D, N), dtype=np.int64)
    ratio = pack['residuals'] / scale[None, None, :]
    selected = ratio.argmax(axis=-1)
    offset = 1 if family == 'residual_mask_tuned' else 1 + ratio.shape[-1]
    return np.where(ratio.max(axis=-1) > settings['threshold'], selected + offset, 0)




STAGES = {'prepared': PROJECT / 'processed' / 'v1',
          'nb02': PROJECT / 'outputs' / 'notebook02_v1',
          'nb03': PROJECT / 'outputs' / 'notebook03_v1'}
INVENTORY_NAMES = {'prepared': 'artifact_inventory.json',
                   'nb02': 'notebook02_artifact_inventory.json',
                   'nb03': 'notebook03_artifact_inventory.json'}
inventories, inventory_hashes = {}, {}
for stage, root in STAGES.items():
    path = root / INVENTORY_NAMES[stage]
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Finish its earlier notebook first.')
    inventory_hashes[stage] = file_sha256(path)
    records = json.loads(path.read_text())
    inventories[stage] = {record['filename']: record for record in records}
    assert len(inventories[stage]) == len(records), 'Duplicate inventory paths.'


def source_file(stage, relative):
    assert not Path(relative).is_absolute() and '..' not in Path(relative).parts
    if relative not in inventories[stage]:
        raise KeyError(f'Missing inventoried input: {stage}/{relative}')
    expected = inventories[stage][relative]['sha256']
    target = LOCAL / 'inputs' / stage / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and file_sha256(target) == expected:
        return target
    temporary = target.with_name(target.name + '.partial')
    shutil.copyfile(STAGES[stage] / relative, temporary)
    assert file_sha256(temporary) == expected, f'Input hash mismatch: {stage}/{relative}'
    os.replace(temporary, target)
    return target


def source_json(stage, relative):
    return json.loads(source_file(stage, relative).read_text())


protocol = source_json('prepared', 'protocol.json')
assert protocol['development_fault_types'] == ['dropout', 'bias', 'gain']
assert protocol['held_out_fault_type'] == 'stuck'
assert protocol['normalization_fit_split'] == 'train'
assert protocol['fault_severities']['stuck']['time_fraction'] == [0.25, 0.5, 1.0]
completion = json.loads((STAGES['prepared'] / 'notebook01_complete.json').read_text())
assert completion['ready_for_training']
assert completion['artifact_inventory_sha256'] == inventory_hashes['prepared']
assert completion['protocol_sha256'] == inventories['prepared']['protocol.json']['sha256']
report2 = source_json('nb02', 'notebook02_run_report.json')
report3 = source_json('nb03', 'notebook03_run_report.json')
for report in [report2, report3]:
    assert report['datasets_completed'] == ['dsads', 'har']
    assert report['fold'] == FOLD and report['seed'] == SEED
    assert report['test_predictions_computed'] == 0
    assert report['held_out_faults_evaluated'] == []

CONDITIONS = [('clean', None, None)] + [(f'{fault}_{i+1}', fault, i)
             for fault in ['dropout', 'bias', 'gain', 'stuck'] for i in range(3)]
SCOPES = {'clean': ['clean'],
          'known_faults': [name for name, fault, _ in CONDITIONS if fault in ['dropout', 'bias', 'gain']],
          'unseen_stuck': [name for name, fault, _ in CONDITIONS if fault == 'stuck']}
frozen_datasets = {}
for dataset in ['dsads', 'har']:
    run = f'{dataset}_{FOLD}_seed_{SEED}'
    config2 = source_json('nb02', run + '/run_configuration.json')
    config3 = source_json('nb03', run + '/run_configuration.json')
    done2 = source_json('nb02', run + '/dataset_complete.json')
    selected = source_json('nb03', run + '/selected_policy.json')
    assert selected['signature'] == config3['signature']
    assert done2['signature'] == config2['signature']
    assert config2['settings']['protocol_sha256'] == completion['protocol_sha256']
    for filename, digest in config2['settings']['input_sha256'].items():
        assert inventories['prepared'][filename]['sha256'] == digest
    assert config3['configuration']['sklearn_version'] == sklearn.__version__, (
        'The scikit-learn version differs from Notebook 03. Restore that version before loading its saved policy models: '
        + config3['configuration']['sklearn_version'])
    for record in config3['configuration']['validation_inputs']:
        assert inventories['nb02'][record['filename']]['sha256'] == record['sha256']
    needed = [('nb02', run + '/' + name) for name in ['classifier_best.pt', 'reconstructor_best.pt']]
    needed += [('nb02', run + '/action_cache/validation__' + name + '.npz') for name in ['clean', 'bias_3']]
    needed += [('nb03', run + '/' + name) for name in
               ['benefit_head.joblib', 'harm_head.joblib', 'selected_policy.json', 'validation_predictions.npz']]
    for stage, name in needed:
        source_file(stage, name)
    baseline = report3['findings'][dataset]['strongest_baseline_by_macro_f1']
    frozen_datasets[dataset] = {'run': run, 'selected': selected, 'primary_comparator': baseline,
        'classifier_signature': config2['signature'], 'policy_signature': config3['signature'],
        'old_cache_signature': done2['cache_signature'],
        'input_fingerprints': [{'stage': stage, 'filename': name, 'sha256': inventories[stage][name]['sha256']}
                               for stage, name in needed],
        'prepared_fingerprints': config2['settings']['input_sha256']}

# Persist the complete evaluation plan BEFORE reading raw signal arrays or making test predictions.
plan = {'code_version': CODE_VERSION, 'cell_signature': CELL_SIGNATURE, 'fold': FOLD, 'seed': SEED,
        'datasets': frozen_datasets, 'source_inventory_hashes': inventory_hashes,
        'conditions': CONDITIONS, 'scopes': SCOPES, 'fault_specification': protocol['fault_severities'],
        'fault_rng': '2026 + crc32(dataset|test|condition)',
        'inference_batch_size': INFERENCE_BATCH_SIZE,
        'environment': {'torch': str(torch.__version__), 'numpy': np.__version__, 'sklearn': sklearn.__version__,
                        'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'},
        'primary_comparator_selection': 'strongest non-oracle baseline on prior validation, frozen before test',
        'selection_or_refitting_on_test': False, 'significance_tests': False,
        'study_scope': 'fold_0, seed 2026 confirmation; remaining participant folds and repeated seeds pending',
        'aggregation': 'Macro-F1 over all test windows per condition; equal-weight mean within each fault scope',
        'repeated_fault_views_are_independent_samples': False,
        'participant_reporting': 'per-subject Macro-F1 over classes present for that subject; no population CI'}
FROZEN_SIGNATURE = json_signature(plan)
freeze_path = OUTPUT / 'notebook04_frozen_manifest.json'
if freeze_path.exists():
    existing = json.loads(freeze_path.read_text())
    assert existing['signature'] == FROZEN_SIGNATURE, (
        'Frozen evaluation inputs, code or environment changed. Do not overwrite this test run or retune on test results.')
else:
    save_json(freeze_path, {'signature': FROZEN_SIGNATURE, 'plan': plan})
print('Evaluation plan frozen before test inference:', FROZEN_SIGNATURE[:16], flush=True)


def pack_from_caches(paths, expected_signature):
    matrices, residuals, statistics = [], [], []
    anchor, conditions = None, []
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == expected_signature
            identity = {key: saved[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key, value in identity.items():
                    assert np.array_equal(value, anchor[key]), f'Cache alignment error: {key}'
            matrices.append(saved['probabilities'])
            residuals.append(saved['reconstruction_residuals'])
            statistics.append(saved['observed_group_statistics'])
            conditions.append(str(saved['condition'].item()))
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and len(np.unique(pack['window_ids'])) == N
    return pack


def score_policy(pack, selected, benefit_model, harm_model):
    scale = np.asarray(selected['residual_scale'], dtype=np.float64)
    assert scale.ndim == 1 and np.all(scale > 0) and np.isfinite(scale).all()
    features, feature_names = policy_features(pack['probabilities'], pack['residuals'], pack['statistics'], scale)
    assert feature_names == selected['feature_names']
    assert pack['action_names'].tolist() == selected['action_names']
    shape = pack['probabilities'].shape[:2] + (pack['probabilities'].shape[2] - 1,)
    fix = predict_head(benefit_model, features).reshape(shape)
    harm = predict_head(harm_model, features).reshape(shape)
    assert np.isfinite(fix).all() and np.isfinite(harm).all()
    assert np.all((fix >= 0) & (fix <= 1)) and np.all((harm >= 0) & (harm <= 1))
    return fix, harm, scale


def check_validation_parity(dataset, raw, splits, mean, std, groups, classes, classifier, reconstructor, benefit, harm):
    info = frozen_datasets[dataset]
    selected, run = info['selected'], info['run']
    paths = [source_file('nb02', run + f'/action_cache/validation__{c}.npz') for c in ['clean', 'bias_3']]
    pack = pack_from_caches(paths, info['old_cache_signature'])
    assert np.array_equal(pack['source_indices'], splits['validation'])
    validation = np.ascontiguousarray((raw[splits['validation']] - mean[None, :, None]) / std[None, :, None], dtype=np.float32)
    # Recreate faults over the WHOLE split so RNG draws match the original cache.
    indices = np.unique(np.linspace(0, len(validation)-1, min(32, len(validation))).astype(int))
    max_errors = {}
    for d, condition in enumerate(pack['conditions']):
        token = zlib.crc32(f'{dataset}|validation|{condition}'.encode())
        observed = validation if condition == 'clean' else corrupt_signals(validation, groups, np.random.default_rng(SEED + token), 'bias', 2)
        p, r, s, _ = action_outputs(classifier, reconstructor, observed[indices], groups, classes)
        for label, actual, expected in [('probabilities', p, pack['probabilities'][d, indices]),
                                        ('residuals', r, pack['residuals'][d, indices]),
                                        ('statistics', s, pack['statistics'][d, indices])]:
            error = float(np.max(np.abs(actual-expected)))
            max_errors[condition + '_' + label] = error
            assert np.allclose(actual, expected, atol=2e-5, rtol=2e-5), (
                f'{dataset} validation {condition}/{label} failed parity (max abs error {error}). Stop before test.')
    fix, risk, scale = score_policy(pack, selected, benefit, harm)
    with np.load(source_file('nb03', run + '/validation_predictions.npz'), allow_pickle=False) as old:
        assert old['signature'].item() == info['policy_signature']
        assert np.array_equal(old['window_ids'], pack['window_ids'])
        d_indices = [old['conditions'].tolist().index(c) for c in pack['conditions']]
        assert np.allclose(fix, old['probability_fix'][d_indices], atol=1e-6, rtol=1e-6)
        assert np.allclose(risk, old['probability_harm'][d_indices], atol=1e-6, rtol=1e-6)
        for family, item in selected['selected_methods'].items():
            choices = choose_actions(pack, fix, risk, scale, item['settings'])
            m = old['methods'].tolist().index(family)
            assert np.array_equal(choices, old['choices'][m, d_indices]), 'Saved policy decision parity failed.'
    del pack, validation, observed
    print('  Validation parity passed: neural outputs, policy probabilities and frozen decisions.', flush=True)
    return max_errors


def scope_summary(rows):
    result = []
    frame = pd.DataFrame(rows)
    for method in frame.method.unique():
        for scope, conditions in SCOPES.items():
            part = frame[(frame.method == method) & frame.condition.isin(conditions)]
            assert len(part) == len(conditions)
            result.append({'method': method, 'scope': scope, 'conditions': len(conditions),
                'unique_test_windows': int(part.windows.iloc[0]), **{key: float(part[key].mean()) for key in
                ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}})
    return result


def per_subject_results(pack, choices):
    # HAR individual participants need not contain every activity. Restrict to
    # that participant's observed classes for descriptive participant scores.
    result = []
    all_predictions = pack['probabilities'].argmax(axis=-1)
    K = pack['probabilities'].shape[-1]
    for method, actions in choices.items():
        if method == 'oracle_diagnostic':
            continue
        predictions = np.take_along_axis(all_predictions, actions[..., None], axis=2).squeeze(2)
        for subject in np.unique(pack['subjects']):
            use = pack['subjects'] == subject
            y = pack['y'][use]
            present = np.unique(y)
            for scope, conditions in SCOPES.items():
                scores, accuracies, helpful, harmful, interventions = [], [], [], [], []
                for name in conditions:
                    d = pack['conditions'].index(name)
                    pred = predictions[d, use]
                    cm = np.bincount(y*K+pred, minlength=K*K).reshape(K,K)
                    denominator = cm.sum(0) + cm.sum(1)
                    f1s = np.divide(2.0*cm.diagonal(), denominator, out=np.zeros(K), where=denominator>0)
                    correct = pred == y
                    original_correct = all_predictions[d, use, 0] == y
                    scores.append(float(f1s[present].mean()))
                    accuracies.append(float(correct.mean()))
                    helpful.append(float((~original_correct & correct).mean()))
                    harmful.append(float((original_correct & ~correct).mean()))
                    interventions.append(float((actions[d, use] != 0).mean()))
                result.append({'subject': int(subject), 'method': method, 'scope': scope,
                    'windows': int(use.sum()), 'classes_present': len(present),
                    'macro_f1_present_classes': float(np.mean(scores)), 'accuracy': float(np.mean(accuracies)),
                    'helpful_rate': float(np.mean(helpful)), 'harmful_rate': float(np.mean(harmful)),
                    'intervention_rate': float(np.mean(interventions))})
    return result


def run_dataset(dataset):
    print('\n' + dataset.upper() + ' | loading frozen models', flush=True)
    info = frozen_datasets[dataset]
    selected, run_name = info['selected'], info['run']
    run = OUTPUT / run_name
    (run / 'action_cache').mkdir(parents=True, exist_ok=True)
    with np.load(source_file('prepared', f'{dataset}_signals.npz'), allow_pickle=False) as saved:
        raw, y, subjects, window_ids = saved['X'], saved['y'], saved['subjects'], saved['window_ids']
    with np.load(source_file('prepared', f'{dataset}_splits.npz'), allow_pickle=False) as saved:
        splits = {split: saved[f'{FOLD}_{split}'] for split in ['train', 'policy', 'validation', 'test']}
    with np.load(source_file('prepared', f'{dataset}_{FOLD}_normalization.npz'), allow_pickle=False) as saved:
        mean, std = saved['mean'], saved['std']
    groups = protocol['datasets'][dataset]['sensor_groups']
    classes = len(protocol['datasets'][dataset]['class_names'])
    assert np.array_equal(np.unique(y), np.arange(classes))
    for split, indices in splits.items():
        assert len(indices) == len(np.unique(indices)) and np.all((indices>=0) & (indices<len(raw)))
        for other, second in splits.items():
            if split != other:
                assert not set(subjects[indices].tolist()) & set(subjects[second].tolist()), 'Participant overlap.'
    assert len(np.concatenate(list(splits.values()))) == len(raw)
    assert len(np.unique(np.concatenate(list(splits.values())))) == len(raw)
    classifier = ActivityCNN(raw.shape[1], classes).to(DEVICE)
    reconstructor = SensorReconstructor(raw.shape[1]).to(DEVICE)
    for name, model in [('classifier', classifier), ('reconstructor', reconstructor)]:
        checkpoint = torch.load(source_file('nb02', run_name + f'/{name}_best.pt'), map_location='cpu', weights_only=True)
        assert checkpoint['signature'] == info['classifier_signature']
        model.load_state_dict(checkpoint['model'], strict=True)
        model.eval()
        del checkpoint
    heads = []
    for name in ['benefit', 'harm']:
        payload = joblib.load(source_file('nb03', run_name + f'/{name}_head.joblib'))
        assert payload['signature'] == info['policy_signature']
        assert payload['feature_names'] == selected['feature_names']
        heads.append(payload['model'])
    parity = check_validation_parity(dataset, raw, splits, mean, std, groups, classes,
                                     classifier, reconstructor, *heads)
    indices = splits['test']
    values = np.ascontiguousarray((raw[indices] - mean[None, :, None]) / std[None, :, None], dtype=np.float32)
    del raw
    action_names = np.asarray(['keep'] + [f'mask:{g["name"]}' for g in groups] + [f'reconstruct:{g["name"]}' for g in groups])
    assert action_names.tolist() == selected['action_names']
    paths, timings, cache_hashes = [], [], []
    for condition, fault, severity in CONDITIONS:
        path = run / 'action_cache' / f'test__{condition}.npz'
        hash_path = path.with_name(path.name + '.sha256.json')
        if path.exists() and hash_path.exists():
            digest = json.loads(hash_path.read_text())
            assert digest['signature'] == FROZEN_SIGNATURE and digest['sha256'] == file_sha256(path), 'Test cache hash mismatch.'
            with np.load(path, allow_pickle=False) as cached:
                assert cached['signature'].item() == FROZEN_SIGNATURE
                for key, expected in [('window_ids', window_ids[indices]), ('source_indices', indices),
                                      ('y', y[indices]), ('subjects', subjects[indices]), ('action_names', action_names)]:
                    assert np.array_equal(cached[key], expected), 'Test cache identity mismatch.'
                assert cached['condition'].item() == condition and cached['split'].item() == 'test'
                seconds = float(cached['full_candidate_inference_seconds'].item())
            status = 'reused'
        else:
            if path.exists():
                print(f'  Recovering interrupted {condition} cache from the same frozen inputs.', flush=True)
            token = zlib.crc32(f'{dataset}|test|{condition}'.encode())
            observed = values if fault is None else corrupt_signals(values, groups, np.random.default_rng(SEED+token), fault, severity)
            p, r, s, seconds = action_outputs(classifier, reconstructor, observed, groups, classes)
            save_npz(path, signature=np.asarray(FROZEN_SIGNATURE), probabilities=p, reconstruction_residuals=r,
                     observed_group_statistics=s, y=y[indices], subjects=subjects[indices], window_ids=window_ids[indices],
                     source_indices=indices, condition=np.asarray(condition), split=np.asarray('test'), action_names=action_names,
                     full_candidate_inference_seconds=np.asarray(seconds),
                     generation_device=np.asarray(torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'))
            save_json(hash_path, {'signature': FROZEN_SIGNATURE, 'sha256': file_sha256(path)})
            del observed, p, r, s
            status = 'saved'
        paths.append(path)
        cache_hashes.append({'filename': str(path.relative_to(OUTPUT)), 'sha256': file_sha256(path)})
        timings.append({'dataset': dataset, 'condition': condition, 'unique_windows': len(indices),
                        'candidate_actions': len(action_names), 'candidate_generation_seconds': seconds})
        print(f'  test {condition:10s}: {len(indices)} windows | {status} | candidate inference {seconds:.1f}s', flush=True)
    del values, classifier, reconstructor
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    pack = pack_from_caches(paths, FROZEN_SIGNATURE)
    started = time.perf_counter()
    fix, harm, scale = score_policy(pack, selected, *heads)
    head_seconds = time.perf_counter()-started
    choices = baseline_choices(pack, scale)
    for family, item in selected['selected_methods'].items():
        choices[family] = choose_actions(pack, fix, harm, scale, item['settings'])
    assert info['primary_comparator'] in choices and info['primary_comparator'] != 'oracle_diagnostic'
    # A threshold of 2 for P(fix)-lambda*P(harm), lambda>=0, is a structural keep rule.
    settings = selected['selected_methods']['harm_aware_policy']['settings']
    structurally_keep = settings['harm_penalty'] >= 0 and settings['threshold'] >= 1.0
    if structurally_keep:
        assert np.all(choices['harm_aware_policy'] == 0)
    rows = [{'dataset': dataset, **row} for method, actions in choices.items()
            for row in evaluate_choices(pack, actions, method)]
    summary = [{'dataset': dataset, **row} for row in scope_summary(rows)]
    subjects_rows = [{'dataset': dataset, **row} for row in per_subject_results(pack, choices)]
    methods = list(choices)
    action_predictions = pack['probabilities'].argmax(axis=-1)
    predictions = np.stack([np.take_along_axis(action_predictions, choices[m][...,None], axis=2).squeeze(2) for m in methods])
    save_npz(run / 'test_predictions.npz', signature=np.asarray(FROZEN_SIGNATURE), methods=np.asarray(methods),
             conditions=np.asarray(pack['conditions']), predictions=predictions,
             choices=np.stack([choices[m] for m in methods]), y=pack['y'], subjects=pack['subjects'],
             window_ids=pack['window_ids'], source_indices=pack['source_indices'], probability_fix=fix, probability_harm=harm)
    lookup = {(row['method'],row['scope']): row for row in summary}
    comparisons = []
    for scope in SCOPES:
        learned = lookup['harm_aware_policy',scope]
        baseline = lookup[info['primary_comparator'],scope]
        keep = lookup['keep',scope]
        comparisons.append({'dataset':dataset,'scope':scope,'frozen_comparator':info['primary_comparator'],
            'keep_macro_f1':keep['macro_f1'],'comparator_macro_f1':baseline['macro_f1'],
            'policy_macro_f1':learned['macro_f1'],'delta_macro_f1_pp':100*(learned['macro_f1']-baseline['macro_f1']),
            'policy_intervention_rate':learned['intervention_rate'],'policy_helpful_rate':learned['helpful_rate'],
            'policy_harmful_rate':learned['harmful_rate'], 'comparator_harmful_rate':baseline['harmful_rate']})
    clean = lookup['harm_aware_policy','clean']
    clean_keep = lookup['keep','clean']
    guards = {'clean_f1_loss_pp':100*(clean_keep['macro_f1']-clean['macro_f1']),
              'clean_f1_loss_le_0_5pp':bool(clean['macro_f1']>=clean_keep['macro_f1']-.005-1e-12),
              'clean_harm_le_1pct':bool(clean['harmful_rate']<=.01+1e-12),
              'known_fault_harm_le_1pct':bool(lookup['harm_aware_policy','known_faults']['harmful_rate']<=.01+1e-12),
              'unseen_stuck_harm_le_1pct':bool(lookup['harm_aware_policy','unseen_stuck']['harmful_rate']<=.01+1e-12),
              'note':'Descriptive test checks; never used to change the policy; no risk guarantee.'}
    result = {'dataset':dataset,'signature':FROZEN_SIGNATURE,'test_windows':len(indices),
              'test_participants':[int(x) for x in np.unique(pack['subjects'])],
              'frozen_primary_comparator':info['primary_comparator'], 'selected_policy':settings,
              'structurally_keep_only':bool(structurally_keep),'validation_parity_max_abs_errors':parity,
              'test_empirical_guards':guards,'comparisons':comparisons,
              'policy_feature_and_head_scoring_seconds':head_seconds,
              'head_scoring_excludes_candidate_generation':True,
              'cache_hashes':cache_hashes,'conditions_evaluated':pack['conditions']}
    save_json(run / 'dataset_complete.json', result)
    gc.collect()
    return rows, summary, subjects_rows, comparisons, timings, result


def main():
    rows, summary, participant_rows, comparisons, timings, results = [], [], [], [], [], []
    for dataset in ['dsads','har']:
        a,b,c,d,e,f = run_dataset(dataset)
        rows.extend(a); summary.extend(b); participant_rows.extend(c)
        comparisons.extend(d); timings.extend(e); results.append(f)
        save_csv(OUTPUT / 'notebook04_test_summary.csv', pd.DataFrame(summary))
        save_csv(OUTPUT / 'notebook04_per_condition.csv', pd.DataFrame(rows))
        save_csv(OUTPUT / 'notebook04_per_participant.csv', pd.DataFrame(participant_rows))
        save_csv(OUTPUT / 'notebook04_primary_comparison.csv', pd.DataFrame(comparisons))
        save_csv(OUTPUT / 'notebook04_inference_cost.csv', pd.DataFrame(timings))
    report = {'code_version':CODE_VERSION,'signature':FROZEN_SIGNATURE,'datasets_completed':['dsads','har'],
              'fold':FOLD,'seed':SEED,'evaluation_split':'reserved participant test partitions',
              'new_models_fit':0,'thresholds_retuned':0,'validation_parity_passed':True,
              'held_out_faults_evaluated':['stuck'],'conditions_per_dataset':len(CONDITIONS),
              'test_predictions_computed':sum(item['test_windows']*len(CONDITIONS) for item in results),
              'prediction_count_unit':'original-window x condition; not independent samples',
              'repeated_seeds_and_remaining_dsads_folds_completed':False,
              'significance_or_population_generalization_claim':False,
              'timing_note':'Full candidate generation is required by this implementation. Low intervention rate does not imply low inference cost.',
              'future_revision_note':'After viewing these results, this test evaluation is no longer fresh evidence for a revised method.',
              'results':results,'output_directory':str(OUTPUT)}
    save_json(OUTPUT / 'notebook04_run_report.json', report)
    inventory = [{'filename':str(p.relative_to(OUTPUT)),'bytes':p.stat().st_size,'sha256':file_sha256(p)}
        for p in sorted(OUTPUT.rglob('*')) if p.is_file() and not p.name.endswith('.partial')
        and p.name!='notebook04_artifact_inventory.json']
    save_json(OUTPUT / 'notebook04_artifact_inventory.json', inventory)
    frame = pd.DataFrame(comparisons)
    display = frame.copy()
    for col in ['keep_macro_f1','comparator_macro_f1','policy_macro_f1','policy_intervention_rate',
                'policy_helpful_rate','policy_harmful_rate','comparator_harmful_rate']:
        display[col] *= 100.0
    print('\nNOTEBOOK 04 COMPLETE')
    print('FROZEN TEST CONFIRMATION | percentages; delta in percentage points')
    print(display.to_string(index=False,float_format=lambda x:f'{x:.2f}'))
    for result in results:
        print(f"{result['dataset'].upper()}: {result['test_windows']} unique test windows; "
              f"{len(result['test_participants'])} participants; structural keep-only={result['structurally_keep_only']}")
        print('  Observed test checks:', json.dumps(result['test_empirical_guards']))
    print('New training runs: 0 | Threshold changes: 0 | Known faults: 9 | Unseen stuck conditions: 3')
    print('One fold and one seed only. Full repeated-seed / remaining-fold evaluation is still pending.')
    print('Outputs:', OUTPUT)
    print('Share this final output and notebook04_test_summary.csv, notebook04_run_report.json, notebook04_per_participant.csv.')


with threadpool_limits(limits=2):
    main()


Mounted at /content/drive
NOTEBOOK 04 | frozen fold_0 test confirmation | one executable cell
Device: Tesla T4 | PyTorch: 2.11.0+cu130
No training or threshold selection. Test participants; known faults and held-out stuck faults reported separately.
Evaluation plan frozen before test inference: 5e99341b92e44d9f

DSADS | loading frozen models
  Validation parity passed: neural outputs, policy probabilities and frozen decisions.
  test clean     : 2280 windows | saved | candidate inference 0.7s
  test dropout_1 : 2280 windows | saved | candidate inference 0.6s
  test dropout_2 : 2280 windows | saved | candidate inference 0.6s
  test dropout_3 : 2280 windows | saved | candidate inference 0.6s
  test bias_1    : 2280 windows | saved | candidate inference 0.6s
  test bias_2    : 2280 windows | saved | candidate inference 0.6s
  test bias_3    : 2280 windows | saved | candidate inference 0.6s
  test gain_1    : 2280 windows | saved | candidate inference 0.6s
  test gain_2    : 2280 windows |